# 4-Class Soccer Event Spotting — Multi-Backbone Comparison (Colab)

**Models:** R2Plus1D-18, R3D-18, MC3-18 (all pretrained Kinetics-400)

**Classes:** background (0), shots off target (1), shots on target (2), goal (3)

**Pipeline overview**
1. Mount Google Drive and install dependencies
2. Select backbone via `MODEL_NAME` in the config cell
3. Load SoccerNet official train/valid/test splits
4. Parse `Labels-v2.json` to extract per-event timestamps with class IDs
5. Build a `ClipDataset` — goals → class 3, shots_on → class 2, shots_off → class 1, random → class 0
6. Train the selected backbone with 4-output head + `CrossEntropyLoss`
7. Sliding-window inference returns softmax over 4 classes per window
8. Per-class NMS + evaluation

Run each backbone by changing `MODEL_NAME` and re-executing from cell 2 onwards.

In [ ]:
# 1. Setup — mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q soccernet wandb opencv-python scikit-learn python-dotenv tqdm

In [ ]:
# 2. Config
# Change MODEL_NAME to switch backbones: 'r2plus1d_18', 'r3d_18', 'mc3_18'
import os

MODEL_NAME  = "r2plus1d_18"   # <-- change this to swap backbone

PROJECT_DIR    = "/content/drive/MyDrive/aspotting"
DATA_DIR       = f"{PROJECT_DIR}/dataset"
CKPT_DIR       = f"{PROJECT_DIR}/checkpoints/{MODEL_NAME}_4class"
RESUME_FROM    = ""   # path to a checkpoint to resume from, or "" for fresh start
MANIFEST_DIR   = f"{PROJECT_DIR}/manifests/4class"
CLIP_CACHE_DIR = f"{PROJECT_DIR}/clip_cache/e2_center"   # reuse existing cache
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)
os.makedirs(CLIP_CACHE_DIR, exist_ok=True)

FPS         = 25
CLIP_SEC    = 4.0
CLIP_FRAMES = 16
CLIP_SIZE   = (112, 112)

BATCH_SIZE   = 16
EPOCHS       = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-3

# Class definitions
NUM_CLASSES = 4
CLASS_NAMES = ["background", "shots_off", "shots_on", "goal"]
# 0 = background, 1 = shots off target, 2 = shots on target (+ penalty), 3 = goal

SHOT_ON_LABELS  = {"shots on target", "penalty"}
SHOT_OFF_LABELS = {"shots off target"}

# Sampling ratios per goal event in the same half
SHOTS_ON_PER_GOAL    = 2   # class-2 clips per goal
SHOTS_OFF_PER_GOAL   = 2   # class-1 clips per goal
NEG_RAND_PER_GOAL    = 4   # class-0 clips per goal
NEG_PER_NO_GOAL_HALF = 12  # class-0 clips for halves with no goals
MIN_NEG_FROM_GOAL_SEC = 12.0

# Manifest
FORCE_REBUILD_MANIFEST = True   # set False to reuse saved CSVs
USE_JITTER             = False   # ±2s offsets for positives

# Clip cache (reuses e2_center cache — clips are class-agnostic)
USE_CLIP_CACHE   = True
USE_AUGMENTATION = True

# Model
FREEZE_BACKBONE = True   # freeze stem + layer1-3, only train layer4 + fc
DROPOUT_P       = 0.4

STRIDE_S      = 2.0
NMS_RADIUS_S  = 10.0
CONF_THRESH   = 0.5

TOLERANCES    = [5, 10, 30, 60]

DEVICE = "cuda" if __import__('torch').cuda.is_available() else "cpu"
print(f"Device         : {DEVICE}")
print(f"Model backbone : {MODEL_NAME}")
print(f"Classes        : {CLASS_NAMES}")
print(f"Clip cache     : {USE_CLIP_CACHE}  |  Augmentation: {USE_AUGMENTATION}")
print(f"Freeze backbone: {FREEZE_BACKBONE}  |  Dropout: {DROPOUT_P}")
print(f"Jitter         : {USE_JITTER}  |  Resume from: {RESUME_FROM or '(none)'}")

In [ ]:
# 3. Imports
import json
import random
import hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import (
    r2plus1d_18, R2Plus1D_18_Weights,
    r3d_18,      R3D_18_Weights,
    mc3_18,      MC3_18_Weights,
)
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm.auto import tqdm
import wandb

# W&B API key — set via Colab Secrets (Secrets tab) or directly below
# Colab Secrets: add WANDB_API_KEY and WANDB_PROJECT
try:
    from google.colab import userdata
    os.environ.setdefault('WANDB_API_KEY', userdata.get('WANDB_API_KEY'))
    os.environ.setdefault('WANDB_PROJECT', userdata.get('WANDB_PROJECT'))
except Exception:
    pass  # set os.environ['WANDB_API_KEY'] and os.environ['WANDB_PROJECT'] manually if needed

try:
    from SoccerNet.Downloader import getListGames
except ImportError:
    raise ImportError("Run the setup cell above first.")

# Pretrained weights URLs for reference
MODEL_URLS = {
    "r2plus1d_18": R2Plus1D_18_Weights.KINETICS400_V1.url,
    "r3d_18":      R3D_18_Weights.KINETICS400_V1.url,
    "mc3_18":      MC3_18_Weights.KINETICS400_V1.url,
}
print(f"Pretrained weights URL: {MODEL_URLS[MODEL_NAME]}")
print("Imports OK")

In [ ]:
# 4. SoccerNet official splits
all_splits = {}
for split in ("train", "valid", "test"):
    games = getListGames(split)
    local = [g for g in games if Path(DATA_DIR, g, "Labels-v2.json").exists()]
    all_splits[split] = local
    print(f"{split:5s}  total={len(games):3d}  local={len(local):3d}")

In [ ]:
# 5. Annotation parser — 4-class
# Returns all labelled events per half with integer class IDs.
# Background (class 0) has no annotations — it is sampled randomly.

def parse_annotations(game_rel_path):
    """
    Returns:
        events : {half: [(seconds, class_id), ...]}
            class_id: 1=shots_off, 2=shots_on, 3=goal  (0=background never returned)
    """
    json_path = Path(DATA_DIR, game_rel_path, "Labels-v2.json")
    with open(json_path) as f:
        data = json.load(f)

    events = defaultdict(list)

    for ann in data["annotations"]:
        label = ann.get("label", "").strip().lower()
        half_str, _ = ann["gameTime"].split(" - ")
        half  = int(half_str)
        pos_s = int(ann["position"]) / 1000.0

        if label == "goal":
            events[half].append((pos_s, 3))
        elif label in SHOT_ON_LABELS:
            events[half].append((pos_s, 2))
        elif label in SHOT_OFF_LABELS:
            events[half].append((pos_s, 1))
        # all other event types are ignored

    return dict(events)


# Sanity check
sample_game = all_splits["train"][0]
ev = parse_annotations(sample_game)
print(f"Sample game : {sample_game}")
for half, evs in sorted(ev.items()):
    for t, cid in sorted(evs):
        print(f"  half={half}  t={t:.1f}s  class={cid} ({CLASS_NAMES[cid]})")

In [ ]:
# 6. Clip reader and Dataset

MEAN = np.array([0.43216, 0.394666, 0.37645],  dtype=np.float32)
STD  = np.array([0.22803, 0.22145,  0.216989], dtype=np.float32)


def read_clip(video_path, center_sec, n_frames=CLIP_FRAMES,
              clip_sec=CLIP_SEC, size=CLIP_SIZE):
    """
    Sample n_frames frames sparsely across clip_sec, centred on center_sec.
    Returns a (C, T, H, W) float32 normalised tensor, or None on failure.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    fps_v = cap.get(cv2.CAP_PROP_FPS) or FPS

    start_sec   = max(0.0, center_sec - clip_sec / 2)
    start_frame = int(start_sec * fps_v)
    step        = max(1, int(clip_sec * fps_v / n_frames))

    frames = []
    for i in range(n_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame + i * step)
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, size)
        frames.append(frame)
    cap.release()

    while len(frames) < n_frames:
        frames.append(frames[-1] if frames else np.zeros((*size, 3), np.uint8))

    arr = np.stack(frames).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    return torch.from_numpy(arr).permute(3, 0, 1, 2)    # (C, T, H, W)


def clip_cache_path(video_path, center_sec):
    """Deterministic .npy path from (video_path, center_sec) — same hash as e2_center."""
    key = f"{video_path}_{center_sec:.4f}"
    h   = hashlib.md5(key.encode()).hexdigest()
    return Path(CLIP_CACHE_DIR) / f"{h}.npy"


def far_from_all(t, times, min_gap):
    return all(abs(t - x) >= min_gap for x in times)


def build_sample_list(game_list, is_train=True):
    """
    Returns list of (video_path, center_sec, class_id, event_type) tuples.
    class_id: 0=background, 1=shots_off, 2=shots_on, 3=goal
    """
    samples = []
    for game in tqdm(game_list, desc="Building samples"):
        events = parse_annotations(game)
        for half in (1, 2):
            vid = Path(DATA_DIR, game, f"{half}_224p.mkv")
            if not vid.exists():
                continue
            cap   = cv2.VideoCapture(str(vid))
            fps_v = cap.get(cv2.CAP_PROP_FPS) or FPS
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            dur = total / fps_v
            if dur < CLIP_SEC + 1:
                continue

            lo = CLIP_SEC / 2
            hi = dur - CLIP_SEC / 2

            half_events = events.get(half, [])
            goals      = [t for t, c in half_events if c == 3]
            shots_on   = [t for t, c in half_events if c == 2]
            shots_off  = [t for t, c in half_events if c == 1]
            all_event_times = [t for t, _ in half_events]

            jitters = (-2.0, 0.0, 2.0) if (is_train and USE_JITTER) else (0.0,)

            # Goals — class 3
            for g in goals:
                for j in jitters:
                    c = float(np.clip(g + j, lo, hi))
                    samples.append((str(vid), c, 3, "goal"))

            # Shots on target — class 2
            shots_on_cands = [t for t in shots_on
                              if far_from_all(t, goals, MIN_NEG_FROM_GOAL_SEC)]
            n_on = min(len(shots_on_cands), SHOTS_ON_PER_GOAL * max(1, len(goals)))
            chosen_on = random.sample(shots_on_cands, k=n_on) if n_on > 0 else []
            for t in chosen_on:
                samples.append((str(vid), float(np.clip(t, lo, hi)), 2, "shots_on"))

            # Shots off target — class 1
            shots_off_cands = [t for t in shots_off
                               if far_from_all(t, goals, MIN_NEG_FROM_GOAL_SEC)]
            n_off = min(len(shots_off_cands), SHOTS_OFF_PER_GOAL * max(1, len(goals)))
            chosen_off = random.sample(shots_off_cands, k=n_off) if n_off > 0 else []
            for t in chosen_off:
                samples.append((str(vid), float(np.clip(t, lo, hi)), 1, "shots_off"))

            # Background — class 0
            n_rand    = (NEG_RAND_PER_GOAL * len(goals) if goals else NEG_PER_NO_GOAL_HALF)
            forbidden = list(all_event_times)
            added = attempts = 0
            while added < n_rand and attempts < n_rand * 40:
                attempts += 1
                t = random.uniform(lo, hi)
                if far_from_all(t, forbidden, MIN_NEG_FROM_GOAL_SEC):
                    samples.append((str(vid), t, 0, "rand"))
                    forbidden.append(t)
                    added += 1

    return samples


class ClipDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if USE_CLIP_CACHE:
            cache = clip_cache_path(row.video_path, float(row.center_sec))
            if cache.exists():
                clip = torch.from_numpy(
                    np.load(str(cache)).astype(np.float32))
            else:
                clip = read_clip(row.video_path, float(row.center_sec))
                if clip is None:
                    clip = torch.zeros(3, CLIP_FRAMES, *CLIP_SIZE)
        else:
            clip = read_clip(row.video_path, float(row.center_sec))
            if clip is None:
                clip = torch.zeros(3, CLIP_FRAMES, *CLIP_SIZE)
        if self.transform is not None:
            clip = self.transform(clip)
        # label is a class index (long), not a float
        return clip, torch.tensor(int(row.label), dtype=torch.long)


print("Dataset utilities defined")

In [ ]:
# 7. Build or load manifest, then create data loaders

train_csv = f"{MANIFEST_DIR}/train_clips.csv"
valid_csv = f"{MANIFEST_DIR}/valid_clips.csv"

if FORCE_REBUILD_MANIFEST or not (Path(train_csv).exists() and Path(valid_csv).exists()):
    random.seed(42)
    train_samples = build_sample_list(all_splits["train"], is_train=True)
    valid_samples = build_sample_list(all_splits["valid"], is_train=False)

    train_df = pd.DataFrame(train_samples, columns=["video_path", "center_sec", "label", "event_type"])
    valid_df = pd.DataFrame(valid_samples, columns=["video_path", "center_sec", "label", "event_type"])

    train_df.to_csv(train_csv, index=False)
    valid_df.to_csv(valid_csv, index=False)
    print(f"Manifests saved to {MANIFEST_DIR}")
else:
    train_df = pd.read_csv(train_csv)
    valid_df = pd.read_csv(valid_csv)
    print(f"Loaded manifests from {MANIFEST_DIR}")

print("\nTrain class distribution:")
for cid, name in enumerate(CLASS_NAMES):
    n = int((train_df.label == cid).sum())
    print(f"  {name:12s} (class {cid}): {n:,}")

print("\nValid class distribution:")
for cid, name in enumerate(CLASS_NAMES):
    n = int((valid_df.label == cid).sum())
    print(f"  {name:12s} (class {cid}): {n:,}")

import torchvision.transforms.v2 as Tv2

_train_transform = Tv2.RandomHorizontalFlip(p=0.5) if USE_AUGMENTATION else None

train_loader = DataLoader(ClipDataset(train_df, transform=_train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(ClipDataset(valid_df, transform=None),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}  Valid batches: {len(valid_loader)}")
print(f"Augmentation: {'ON (horizontal flip)' if USE_AUGMENTATION else 'OFF'}")

In [ ]:
# 7b. Extract and cache clips as .npy files
# Run this once if new clips in the manifest are not yet cached.
# Clips are class-agnostic — the e2_center cache is shared across all backbones.
# After it finishes set USE_CLIP_CACHE=True in config.

def extract_and_cache_clips(df, desc="Extracting"):
    skipped = extracted = failed = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        out = clip_cache_path(row.video_path, float(row.center_sec))
        if out.exists():
            skipped += 1
            continue
        clip = read_clip(row.video_path, float(row.center_sec))
        if clip is None:
            failed += 1
            continue
        np.save(str(out), clip.numpy().astype(np.float16))
        extracted += 1
    print(f"  extracted={extracted}  skipped={skipped}  failed={failed}")

print(f"Cache dir : {CLIP_CACHE_DIR}")
print(f"Clips to cache: {len(train_df) + len(valid_df):,}")
print()

extract_and_cache_clips(train_df, desc="Train clips")
extract_and_cache_clips(valid_df, desc="Valid clips")

n_cached = len(list(Path(CLIP_CACHE_DIR).glob("*.npy")))
size_gb  = sum(f.stat().st_size for f in Path(CLIP_CACHE_DIR).glob("*.npy")) / 1e9
print(f"\nTotal cached: {n_cached:,} files  ({size_gb:.1f} GB)")
print("Set USE_CLIP_CACHE=True in config to use cache during training.")

In [ ]:
# 8. Model
# Select backbone via MODEL_NAME in config cell.
# All three are pretrained on Kinetics-400 and share the same fc input dim (512).

if MODEL_NAME == "r2plus1d_18":
    backbone = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
elif MODEL_NAME == "r3d_18":
    backbone = r3d_18(weights=R3D_18_Weights.KINETICS400_V1)
elif MODEL_NAME == "mc3_18":
    backbone = mc3_18(weights=MC3_18_Weights.KINETICS400_V1)
else:
    raise ValueError(f"Unknown MODEL_NAME: {MODEL_NAME!r}. Choose from 'r2plus1d_18', 'r3d_18', 'mc3_18'.")

model = backbone

if FREEZE_BACKBONE:
    for name, param in model.named_parameters():
        if not (name.startswith("layer4") or name.startswith("fc")):
            param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(p=DROPOUT_P),
    nn.Linear(model.fc.in_features, NUM_CLASSES)
)
model = model.to(DEVICE)

# Class weights: inverse frequency, normalised so mean weight = 1
class_counts = train_df['label'].value_counts().sort_index()
raw_weights  = 1.0 / class_counts.values.astype(np.float32)
class_weights = torch.tensor(
    raw_weights / raw_weights.mean(), dtype=torch.float32, device=DEVICE
)
print("Class weights:")
for name, w in zip(CLASS_NAMES, class_weights.cpu().tolist()):
    print(f"  {name:12s}: {w:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nBackbone  : {MODEL_NAME}")
print(f"Head      : {model.fc}")
print(f"Params    : {total:,}  (trainable: {trainable:,} / frozen: {total - trainable:,})")

In [ ]:
# 9. Training loop

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for clips, labels in tqdm(loader, leave=False):
            clips  = clips.float().to(DEVICE)
            labels = labels.long().to(DEVICE)
            with torch.amp.autocast(DEVICE):
                logits = model(clips)               # [B, 4] — no squeeze
                loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            preds       = torch.argmax(logits, dim=1)
            total_loss += loss.item() * len(labels)
            correct    += (preds == labels).sum().item()
            total      += len(labels)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    macro_f1   = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
    macro_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    macro_rec  = recall_score(all_labels, all_preds, average='macro',    zero_division=0)
    per_cls_f1 = f1_score(all_labels, all_preds, average=None,
                          zero_division=0, labels=list(range(NUM_CLASSES)))
    return total_loss / total, correct / total, macro_prec, macro_rec, macro_f1, per_cls_f1, all_preds, all_labels


# ── Per-run checkpoint directory ──────────────────────────────────────────────
from datetime import datetime
run_dir     = Path(CKPT_DIR) / datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(run_dir, exist_ok=True)
best_ckpt   = str(run_dir / "best.pt")
latest_ckpt = str(run_dir / "latest.pt")
print(f"Checkpoint dir: {run_dir}")

# ── Resume from checkpoint ────────────────────────────────────────────────────
start_epoch   = 1
best_val_f1   = 0.0
wandb_run_id  = None
if RESUME_FROM:
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optim"])
    scheduler.load_state_dict(ckpt["sched"])
    start_epoch  = ckpt["epoch"] + 1
    best_val_f1  = ckpt.get("best_val_f1", 0.0)
    wandb_run_id = ckpt.get("wandb_run_id", None)
    print(f"Resumed from epoch {ckpt['epoch']}  best_val_f1={best_val_f1:.4f}")

# ── Training config summary ───────────────────────────────────────────────────
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("=" * 60)
print("TRAINING CONFIG")
print("=" * 60)
print(f"  Model          : {MODEL_NAME}")
print(f"  Epochs         : {EPOCHS}  (start={start_epoch})")
print(f"  Batch size     : {BATCH_SIZE}  |  LR: {LR}  |  Weight decay: {WEIGHT_DECAY}")
print(f"  Clip           : {CLIP_SEC}s  {CLIP_FRAMES}f  {CLIP_SIZE[0]}px")
print(f"  Freeze backbone: {FREEZE_BACKBONE}  |  Dropout: {DROPOUT_P}")
print(f"  Augmentation   : {'ON (horizontal flip)' if USE_AUGMENTATION else 'OFF'}")
print(f"  Classes        : {CLASS_NAMES}")
print(f"  Class weights  : {[f'{w:.2f}' for w in class_weights.cpu().tolist()]}")
print(f"  Params         : {trainable:,} trainable / {total:,} total")
print(f"  Train clips    : {len(train_df):,}")
print(f"  Resume from    : {RESUME_FROM or '(none)'}")
print("=" * 60)

# ── W&B experiment ────────────────────────────────────────────────────────────
run_name = (f"{MODEL_NAME}_4class_clip{CLIP_SEC}s_lr{LR}_bs{BATCH_SIZE}_e{EPOCHS}")

run = wandb.init(
    project=os.environ["WANDB_PROJECT"],
    entity='jintonyn-oslomet',
    name=run_name,
    id=wandb_run_id,
    resume="allow",
    config={
        "model_name":            MODEL_NAME,
        "num_classes":           NUM_CLASSES,
        "class_names":           CLASS_NAMES,
        "clip_sec":              CLIP_SEC,
        "clip_frames":           CLIP_FRAMES,
        "clip_size":             CLIP_SIZE[0],
        "batch_size":            BATCH_SIZE,
        "epochs":                EPOCHS,
        "lr":                    LR,
        "weight_decay":          WEIGHT_DECAY,
        "freeze_backbone":       FREEZE_BACKBONE,
        "dropout_p":             DROPOUT_P,
        "use_augmentation":      USE_AUGMENTATION,
        "shots_on_per_goal":     SHOTS_ON_PER_GOAL,
        "shots_off_per_goal":    SHOTS_OFF_PER_GOAL,
        "neg_rand_per_goal":     NEG_RAND_PER_GOAL,
        "min_neg_from_goal_s":   MIN_NEG_FROM_GOAL_SEC,
        "stride_s":              STRIDE_S,
        "nms_radius_s":          NMS_RADIUS_S,
        "conf_thresh":           CONF_THRESH,
        "use_clip_cache":        USE_CLIP_CACHE,
        "n_train_clips":         len(train_df),
        "n_valid_clips":         len(valid_df),
        "device":                DEVICE,
        "resume_from":           RESUME_FROM or None,
        "start_epoch":           start_epoch,
    }
)

# Log manifests
artifact = wandb.Artifact(name="manifests_4class", type="dataset")
artifact.add_file(train_csv, name="train_manifest.csv")
artifact.add_file(valid_csv, name="valid_manifest.csv")
run.log_artifact(artifact)
print("W&B run started:", run.name, " id:", run.id)

# ── Training ──────────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(valid_loader, train=False)
    scheduler.step()

    # tr/va: (loss, acc, macro_prec, macro_rec, macro_f1, per_cls_f1, preds, labels)
    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train loss={tr[0]:.4f} acc={tr[1]:.3f} macroF1={tr[4]:.3f}  |  "
          f"val   loss={va[0]:.4f} acc={va[1]:.3f} macroF1={va[4]:.3f}")
    cls_f1_str = "  ".join(f"{CLASS_NAMES[i]}={va[5][i]:.3f}" for i in range(NUM_CLASSES))
    print(f"  val per-class F1: {cls_f1_str}")

    log_dict = {
        "train/loss": tr[0], "train/acc": tr[1],
        "train/macro_prec": tr[2], "train/macro_rec": tr[3], "train/macro_f1": tr[4],
        "val/loss":   va[0], "val/acc":   va[1],
        "val/macro_prec": va[2], "val/macro_rec": va[3], "val/macro_f1": va[4],
    }
    for i, name in enumerate(CLASS_NAMES):
        log_dict[f"val/f1_{name}"] = va[5][i]
        log_dict[f"train/f1_{name}"] = tr[5][i]

    wandb.log(log_dict, step=epoch)
    wandb.log({
        "val/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=va[7], preds=va[6], class_names=CLASS_NAMES,
        ),
    }, step=epoch)

    torch.save({"epoch": epoch, "model": model.state_dict(),
                "optim": optimizer.state_dict(),
                "sched": scheduler.state_dict(),
                "val_macro_f1": va[4],
                "best_val_f1": best_val_f1,
                "wandb_run_id": run.id,
                "model_name": MODEL_NAME}, latest_ckpt)

    if va[4] > best_val_f1:
        best_val_f1 = va[4]
        torch.save(model.state_dict(), best_ckpt)
        wandb.log({"best_val_macro_f1": best_val_f1}, step=epoch)
        wandb.save(best_ckpt)
        print(f"  -> saved best checkpoint  (val macro F1={best_val_f1:.4f})")

run.finish()
print("Training complete.")
print(f"Checkpoints saved to: {run_dir}")

In [ ]:
# 10. Sliding-window inference

def _moving_average(arr, k):
    if k <= 1:
        return arr
    pad    = k // 2
    padded = np.pad(arr, (pad, pad), mode='edge')
    return np.convolve(padded, np.ones(k, dtype=np.float32) / k, mode='valid')


def sliding_window_inference(video_path, conf_thresh=CONF_THRESH,
                              nms_radius_s=NMS_RADIUS_S, smooth_k=1):
    """
    Slides a window over the half and returns (detections, raw_curve).

    raw_curve: list of (center_sec, [p0, p1, p2, p3]) in chronological order
               where p_i is the softmax probability for class i.
    detections: list of {"timestamp_s": t, "confidence": p, "class_id": c}
                from per-class NMS over all event classes (1, 2, 3).
    """
    cap          = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_v        = cap.get(cv2.CAP_PROP_FPS) or FPS
    cap.release()

    dur       = total_frames / fps_v
    centers_s = np.arange(CLIP_SEC / 2, dur - CLIP_SEC / 2, STRIDE_S)

    model.eval()
    raw           = []
    batch_clips   = []
    batch_centers = []

    def _flush():
        if not batch_clips:
            return
        t = torch.stack(batch_clips).float().to(DEVICE)
        with torch.no_grad(), torch.amp.autocast(DEVICE):
            probs = torch.softmax(model(t), dim=1).cpu().numpy()   # [B, 4]
        for cs, class_probs in zip(batch_centers, probs):
            raw.append((float(cs), class_probs.tolist()))
        batch_clips.clear()
        batch_centers.clear()

    for cs in tqdm(centers_s, desc=Path(video_path).name, leave=False):
        clip = read_clip(video_path, float(cs))
        if clip is None:
            continue
        batch_clips.append(clip)
        batch_centers.append(cs)
        if len(batch_clips) == BATCH_SIZE:
            _flush()
    _flush()

    if not raw:
        return [], []

    raw_curve  = sorted(raw, key=lambda x: x[0])
    detections = postprocess(raw_curve, conf_thresh, nms_radius_s, smooth_k)
    return detections, raw_curve


def postprocess(raw_curve, conf_thresh=CONF_THRESH,
                nms_radius_s=NMS_RADIUS_S, smooth_k=1):
    """
    Apply per-class smoothing + threshold + NMS to a 4-class raw curve.
    Returns detections for event classes 1, 2, 3 (background=0 is not returned).
    """
    if not raw_curve:
        return []

    cs_arr  = [x[0] for x in raw_curve]
    all_dets = []

    for class_id in range(1, NUM_CLASSES):   # skip background (0)
        p_arr = np.array([x[1][class_id] for x in raw_curve], np.float32)
        if smooth_k > 1:
            p_arr = _moving_average(p_arr, smooth_k)

        candidates = sorted(zip(cs_arr, p_arr.tolist()), key=lambda x: -x[1])
        class_dets = []
        for cs, prob in candidates:
            if prob < conf_thresh:
                continue
            if all(abs(cs - d["timestamp_s"]) >= nms_radius_s for d in class_dets):
                class_dets.append({"timestamp_s": cs, "confidence": prob, "class_id": class_id})
        all_dets.extend(class_dets)

    all_dets.sort(key=lambda x: x["timestamp_s"])
    return all_dets


print("Sliding-window inference defined")

In [ ]:
# 10b. Single-half sanity check
# Scores one validation half and prints per-class detections vs ground truth.

SANITY_THRESH   = 0.5
SANITY_SMOOTH_K = 7
SANITY_NMS_S    = 10.0
SANITY_TOL_S    = 5.0

sanity_game = all_splits["valid"][2]
sanity_half = 1
sanity_vid  = Path(DATA_DIR, sanity_game, f"{sanity_half}_224p.mkv")
best_ckpt   = str(sorted(Path(CKPT_DIR).glob("*/best.pt"))[-1])   # most recent

half_events = parse_annotations(sanity_game).get(sanity_half, [])
gt_by_class = {cid: [t for t, c in half_events if c == cid] for cid in range(1, NUM_CLASSES)}

print(f"Game : {sanity_game}")
print(f"Half : {sanity_half}")
for cid in [3, 2, 1]:
    ts = [round(t, 1) for t in gt_by_class.get(cid, [])]
    print(f"  GT {CLASS_NAMES[cid]:12s}: {ts}")
print(f"Threshold={SANITY_THRESH}  smooth_k={SANITY_SMOOTH_K}  nms={SANITY_NMS_S}s\n")

model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

dets, _ = sliding_window_inference(
    str(sanity_vid),
    conf_thresh=SANITY_THRESH,
    nms_radius_s=SANITY_NMS_S,
    smooth_k=SANITY_SMOOTH_K,
)

for class_id in [3, 2, 1]:
    class_name = CLASS_NAMES[class_id]
    gt = gt_by_class.get(class_id, [])
    class_dets = [d for d in dets if d['class_id'] == class_id]

    print(f"--- {class_name} ---")
    print(f"Detected {len(class_dets)} event(s):")
    for d in class_dets:
        t = d['timestamp_s']
        mm, ss  = int(t) // 60, int(t) % 60
        matched = any(abs(t - g) <= SANITY_TOL_S for g in gt)
        print(f"  {mm:02d}:{ss:02d}  conf={d['confidence']:.3f}  {'HIT' if matched else 'FP'}")
    if not class_dets:
        print("  (no detections above threshold)")

    missed = [g for g in gt if not any(abs(g - d['timestamp_s']) <= SANITY_TOL_S for d in class_dets)]
    tp = len(gt) - len(missed)
    fp = sum(1 for d in class_dets if not any(abs(d['timestamp_s'] - g) <= SANITY_TOL_S for g in gt))
    print(f"  Missed: {[round(t, 1) for t in missed]}  TP={tp}  FP={fp}  FN={len(missed)}\n")

In [ ]:
# 11. Run inference on the validation set
import pickle

RAW_CURVES_PATH = f"{PROJECT_DIR}/inference_raw_{MODEL_NAME}_4class.pkl"

model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()
print(f"Loaded: {best_ckpt}")

# all_preds[class_id][(game, half)] = [(timestamp_s, confidence), ...]
all_preds = {cid: {} for cid in range(1, NUM_CLASSES)}
all_gt    = {cid: {} for cid in range(1, NUM_CLASSES)}
all_raw   = {}   # (game, half) -> [(center_sec, [p0,p1,p2,p3]), ...]

for game in tqdm(all_splits["valid"], desc="Inference"):
    half_events_map = parse_annotations(game)
    for half in (1, 2):
        vid = Path(DATA_DIR, game, f"{half}_224p.mkv")
        if not vid.exists():
            continue
        key = (game, half)
        half_events = half_events_map.get(half, [])

        for cid in range(1, NUM_CLASSES):
            all_gt[cid][key] = [t for t, c in half_events if c == cid]

        dets, raw_curve = sliding_window_inference(str(vid))
        all_raw[key] = raw_curve

        for cid in range(1, NUM_CLASSES):
            all_preds[cid][key] = [
                (d["timestamp_s"], d["confidence"])
                for d in dets if d["class_id"] == cid
            ]

with open(RAW_CURVES_PATH, "wb") as f:
    pickle.dump({"all_raw": all_raw, "all_gt": all_gt}, f)

print(f"Done. Halves processed: {len(all_raw)}")
print(f"Raw curves saved to: {RAW_CURVES_PATH}")

In [ ]:
# 11b. Re-run post-processing on saved raw curves (no inference needed)
import pickle

RAW_CURVES_PATH = f"{PROJECT_DIR}/inference_raw_{MODEL_NAME}_4class.pkl"

PP_THRESH   = 0.5
PP_SMOOTH_K = 7
PP_NMS_S    = 10.0

with open(RAW_CURVES_PATH, "rb") as f:
    saved = pickle.load(f)

all_raw = saved["all_raw"]
all_gt  = saved["all_gt"]

all_preds = {cid: {} for cid in range(1, NUM_CLASSES)}
for key, curve in all_raw.items():
    dets = postprocess(curve, PP_THRESH, PP_NMS_S, PP_SMOOTH_K)
    for cid in range(1, NUM_CLASSES):
        all_preds[cid][key] = [
            (d["timestamp_s"], d["confidence"])
            for d in dets if d["class_id"] == cid
        ]

print(f"Re-scored {len(all_raw)} halves  "
      f"(thresh={PP_THRESH}  smooth_k={PP_SMOOTH_K}  nms={PP_NMS_S}s)")

In [ ]:
# 12. Evaluation — per-class Average mAP at multiple tolerances

def compute_ap(preds_with_conf, gt_timestamps, tol):
    preds   = sorted(preds_with_conf, key=lambda x: -x[1])
    matched = set()
    tp_list, fp_list = [], []
    for pred_t, _ in preds:
        best, best_d = None, tol
        for i, gt_t in enumerate(gt_timestamps):
            if i not in matched and abs(pred_t - gt_t) <= best_d:
                best_d = abs(pred_t - gt_t)
                best   = i
        if best is not None:
            matched.add(best); tp_list.append(1); fp_list.append(0)
        else:
            tp_list.append(0); fp_list.append(1)
    if not tp_list or not gt_timestamps:
        return 0.0
    tp_cum = np.cumsum(tp_list)
    fp_cum = np.cumsum(fp_list)
    prec   = tp_cum / (tp_cum + fp_cum + 1e-8)
    rec    = tp_cum / len(gt_timestamps)
    ap, prev_r = 0.0, 0.0
    for p, r in zip(prec, rec):
        ap += p * (r - prev_r); prev_r = r
    return ap


print(f"Results for backbone: {MODEL_NAME}")
header = f"{'Class':>12s}  " + "  ".join(f"mAP@{t}s" for t in TOLERANCES)
print(header)
print("-" * len(header))

for cid in range(1, NUM_CLASSES):
    class_name = CLASS_NAMES[cid]
    aps_per_tol = []
    for tol in TOLERANCES:
        aps = [
            compute_ap(all_preds[cid].get(k, []), all_gt[cid][k], tol)
            for k in all_gt[cid] if all_gt[cid][k]
        ]
        aps_per_tol.append(np.mean(aps) * 100 if aps else 0.0)
    row = f"{class_name:>12s}  " + "  ".join(f"{v:>6.2f}%" for v in aps_per_tol)
    print(row)

In [ ]:
# 13. Single-game demo on test split
import random

demo_game = random.choice(all_splits["test"])
demo_half = 1
demo_vid  = Path(DATA_DIR, demo_game, f"{demo_half}_224p.mkv")

print(f"Game : {demo_game}")
half_events = parse_annotations(demo_game).get(demo_half, [])
for cid in [3, 2, 1]:
    ts = sorted(t for t, c in half_events if c == cid)
    print(f"  GT {CLASS_NAMES[cid]:12s}: {[f'{t:.1f}' for t in ts]}")

if demo_vid.exists():
    dets, _ = sliding_window_inference(str(demo_vid))
    print("\nPredicted events:")
    for d in dets:
        m = int(d['timestamp_s'] // 60)
        s = int(d['timestamp_s'] % 60)
        print(f"  {m:02d}:{s:02d}  class={CLASS_NAMES[d['class_id']]:12s}  conf={d['confidence']:.3f}")
else:
    print("Video not found locally.")